<a href="https://colab.research.google.com/github/k-nikhil-coder/100-day-ai-engineer/blob/main/Day12_kaggle_titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is configured to run with data files (`titanic.csv` and `test.csv`) located in a `data/` subdirectory relative to the notebook. Please ensure these files are present in the specified location within your GitHub repository.

In [10]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import os

# Define URLs for original datasets
train_url_original = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
test_url_original = 'https://raw.githubusercontent.com/agconti/kaggle-titanic/master/data/test.csv'

# Define local paths for the GitHub setup
data_dir = 'data'
train_file_path = os.path.join(data_dir, 'titanic.csv')
test_file_path = os.path.join(data_dir, 'test.csv')

# Create 'data' directory if it doesn't exist and download files for local use in Colab
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    print(f"Directory '{data_dir}' created.")

if not os.path.exists(train_file_path):
    pd.read_csv(train_url_original).to_csv(train_file_path, index=False)
    print(f"Downloaded train data to '{train_file_path}'.")

if not os.path.exists(test_file_path):
    pd.read_csv(test_url_original).to_csv(test_file_path, index=False)
    print(f"Downloaded test data to '{test_file_path}'.")

# 1. Load Titanic Train and Test datasets from local files in a 'data' directory
df = pd.read_csv(train_file_path)

# Separate features and target
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
y = df['Survived']

# 2. Define preprocessing for numerical and categorical data
numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_features = ['Pclass', 'Sex', 'Embarked']
categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

# 3. Combine preprocessing into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

# 4. Build the final Pipeline with a Random Forest Classifier
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'classifier',
            RandomForestClassifier(n_estimators=100, random_state=42),
        ),
    ]
)

# 5. Train the model on the full dataset
model.fit(X, y)
print('Pipeline successfully trained and ready for Kaggle submission!')

# 6. Generate test predictions for Kaggle submission file)
test_df = pd.read_csv(test_file_path)
predictions = model.predict(test_df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']])
submission = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Survived': predictions})
submission.to_csv('my_submission.csv', index=False)

Pipeline successfully trained and ready for Kaggle submission!
